# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ammara-Hussain/flyrank-internship-assignments/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. What one row means (Grain): One row represents the daily performance metrics for a specific content item (content_id / url_hash) on a single calendar date (report_date) belonging to a specific client (client_id).  
2. Table(s) used: fact_content_daily_performance joined with dim_clients and dim_content.  
3. Time window: Monthly partitioned slices (e.g., historical mid-panel training month 2026-03 vs. sealed evaluation month 2026-06).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

1. Target / Label / Proxy: Target $Y$: Predicting whether a content item will achieve high organic traffic next month (e.g., clicks > threshold or relative growth rate in organic clicks).
2. Deliberate Exclusion: Exclude month=2026-06 (_sample table) from feature engineering and training to prevent future-data contamination, keeping it strictly sealed for final testing.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
from google.colab import userdata

# 1. Retrieve the fresh token safely
hf_token = userdata.get('HF_TOKEN')
assert hf_token is not None, "HF_TOKEN secret was not loaded properly!"

# 2. Configure DuckDB connection with HF secret
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

BASE_URL = "hf://datasets/FlyRank/internship-warehouse"

# 3. Run Test Query on dim_clients
df_test = con.sql(f"SELECT * FROM read_parquet('{BASE_URL}/dim_clients.parquet') LIMIT 5").df()
display(df_test)

,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,NaT,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,NaT,NaT
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,NaT
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,NaT


In [3]:
query_grain = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as row_count
FROM read_parquet('{BASE_URL}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1;
"""
df_grain = con.sql(query_grain).df()
print(f"Duplicates found: {len(df_grain)}")  # Must output 0

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicates found: 0


In [4]:
query_stats = f"""
SELECT
    COUNT(*) as total_rows,
    MIN(report_date) as min_date,
    MAX(report_date) as max_date
FROM read_parquet('{BASE_URL}/fact_content_daily_performance/month=2026-03/*.parquet');
"""
df_stats = con.sql(query_stats).df()
display(df_stats)

,total_rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [5]:
query_availability = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(CASE WHEN c.is_active IS TRUE THEN 1 END) as active_rows,
    ROUND(COUNT(CASE WHEN c.is_active IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) as retention_pct
FROM read_parquet('{BASE_URL}/fact_content_daily_performance/month=2026-03/*.parquet') f
JOIN read_parquet('{BASE_URL}/dim_clients.parquet') c ON f.client_hash_id = c.client_hash_id;
"""
df_avail = con.sql(query_availability).df()
display(df_avail)

,total_rows,active_rows,retention_pct
0,9841378,7864344,79.91


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# 1. Five Features Frame
query_features = f"""
SELECT
    content_hash_id,
    AVG(gsc_impressions) as avg_daily_impressions,
    SUM(gsc_clicks) as total_monthly_clicks,
    AVG(gsc_avg_position) as avg_search_position,
    COUNT(DISTINCT report_date) as active_days_count,
    MAX(gsc_clicks / NULLIF(gsc_impressions, 0)) as max_historical_ctr
FROM read_parquet('{BASE_URL}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY content_hash_id;
"""
df = con.sql(query_features).df()

# 2. Target Definition (Future growth outcome)
df['target'] = df['total_monthly_clicks'] * np.random.uniform(0.8, 1.2, len(df))

# 3. Inject Leaked Feature
df['LEAKED_FEATURE'] = df['target'] + np.random.normal(0, 0.01, len(df))

# 4. Test Model WITH Leakage (Artificial high score)
X_leaked = df[['avg_daily_impressions', 'total_monthly_clicks', 'avg_search_position', 'active_days_count', 'max_historical_ctr', 'LEAKED_FEATURE']]
y = df['target']
rf = RandomForestRegressor(n_estimators=50, random_state=42)
rf.fit(X_leaked, y)
print(f"R² WITH Leakage Trap: {r2_score(y, rf.predict(X_leaked)):.4f}")

# 5. Purge Leakage (Honest Baseline Model)
X_honest = df[['avg_daily_impressions', 'total_monthly_clicks', 'avg_search_position', 'active_days_count', 'max_historical_ctr']]
rf.fit(X_honest, y)
print(f"R² WITHOUT Leakage (Honest Baseline): {r2_score(y, rf.predict(X_honest)):.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

R² WITH Leakage Trap: 0.9884
R² WITHOUT Leakage (Honest Baseline): 0.9860


In [14]:
from sklearn.model_selection import train_test_split

# 1. Prepare X and y (Exclude total_monthly_clicks from features so it doesn't cheat)
X_leaked = df[['avg_daily_impressions', 'avg_search_position', 'active_days_count', 'max_historical_ctr', 'LEAKED_FEATURE']]
X_honest = df[['avg_daily_impressions', 'avg_search_position', 'active_days_count', 'max_historical_ctr']]
y = df['target']

# 2. Proper Train/Test Split
X_train_leak, X_test_leak, y_train, y_test = train_test_split(X_leaked, y, test_size=0.2, random_state=42)
X_train_hon, X_test_hon, _, _ = train_test_split(X_honest, y, test_size=0.2, random_state=42)

# 3. Fit and Evaluate Model WITH Leakage
rf_leak = RandomForestRegressor(n_estimators=50, random_state=42)
rf_leak.fit(X_train_leak, y_train)
r2_leak = r2_score(y_test, rf_leak.predict(X_test_leak))
print(f"R² WITH Leakage Trap: {r2_leak:.4f}")

# 4. Fit and Evaluate Model WITHOUT Leakage
rf_hon = RandomForestRegressor(n_estimators=50, random_state=42)
rf_hon.fit(X_train_hon, y_train)
r2_hon = r2_score(y_test, rf_hon.predict(X_test_hon))
print(f"R² WITHOUT Leakage (Honest Baseline): {r2_hon:.4f}")

R² WITH Leakage Trap: 0.9989
R² WITHOUT Leakage (Honest Baseline): 0.6607


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.